
# Water PVT Phase Diagram with Solid–Liquid–Vapor Boundaries

This notebook builds a **schematic PVT phase diagram of water** in Python that includes:

- **solid–vapor** boundary
- **solid–liquid** boundary
- **liquid–vapor** boundary
- **triple point**
- **critical point**
- an interactive **3D Plotly** figure you can rotate

## Important note

A full high-accuracy global PVT phase diagram of water across **solid, liquid, and vapor** is not easy to generate from a single simple property call. In practice:

- **CoolProp** is very convenient for fluid properties and the liquid–vapor boundary
- but a complete solid-region treatment is more limited
- so for teaching and visualization, a very common approach is to build a **schematic phase-boundary model**

This notebook does exactly that:

1. uses **real water constants** for the triple point and critical point,
2. uses **CoolProp** for the liquid–vapor coexistence curve,
3. adds **schematic solid–vapor** and **solid–liquid** boundary curves for visualization.

So this notebook is best viewed as a **teaching-quality 3D phase diagram**, not a full research-grade equation-of-state model for all ice phases.


In [ ]:

# Uncomment if needed
# !pip install CoolProp plotly numpy matplotlib


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from CoolProp.CoolProp import PropsSI


## 1. Basic constants for water

In [ ]:

fluid = "Water"

T_crit = PropsSI("Tcrit", fluid)                 # K
P_crit = PropsSI("Pcrit", fluid)                 # Pa
rho_crit = PropsSI("rhocrit", fluid)             # kg/m^3
v_crit = 1.0 / rho_crit

T_triple = PropsSI("Ttriple", fluid)             # K
P_triple = PropsSI("ptriple", fluid)             # Pa

print(f"Triple point temperature = {T_triple:.5f} K")
print(f"Triple point pressure    = {P_triple:.5f} Pa")
print(f"Critical temperature     = {T_crit:.3f} K")
print(f"Critical pressure        = {P_crit/1e6:.3f} MPa")
print(f"Critical volume          = {v_crit:.6e} m^3/kg")



## 2. Liquid–vapor coexistence curve from CoolProp

For \(T_{\mathrm{triple}} < T < T_c\), we compute:

- saturation pressure
- saturated liquid specific volume
- saturated vapor specific volume


In [ ]:

Tsat = np.linspace(T_triple + 0.01, T_crit - 0.5, 260)

Psat = []
v_liq = []
v_vap = []

for T in Tsat:
    try:
        P = PropsSI("P", "T", float(T), "Q", 0, fluid)
        rho_l = PropsSI("D", "T", float(T), "Q", 0, fluid)
        rho_v = PropsSI("D", "T", float(T), "Q", 1, fluid)

        Psat.append(P)
        v_liq.append(1.0 / rho_l)
        v_vap.append(1.0 / rho_v)
    except Exception:
        pass

Psat = np.array(Psat)
v_liq = np.array(v_liq)
v_vap = np.array(v_vap)

print("Liquid–vapor boundary points:", len(Psat))



## 3. Add schematic solid–vapor and solid–liquid boundaries

### Solid–vapor boundary
Below the triple point, water can coexist as **ice + vapor**.  
For a simple teaching visualization, we model the sublimation curve in a Clausius-like form:

\[
\ln P = A - \frac{B}{T}
\]

and choose constants so that the curve passes through the **triple point**.

### Solid–liquid boundary
For water near ordinary pressures, the melting line has a **negative slope** in \(P\)-\(T\) space.
We build a simple schematic curve passing through the triple point to reflect that unusual behavior.

These are **schematic teaching curves**, not full metrology-grade data.


In [ ]:

# --- solid-vapor (ice-vapor) boundary: schematic sublimation curve ---
T_sv = np.linspace(170.0, T_triple, 220)

# choose B and enforce passage through triple point
B_sv = 5200.0
A_sv = np.log(P_triple) + B_sv / T_triple
P_sv = np.exp(A_sv - B_sv / T_sv)

# schematic specific volumes on the two coexistence branches
# ice is nearly incompressible compared with vapor
v_solid_sv = 1.0 / 920.0 * np.ones_like(T_sv)   # ~ 0.001087 m^3/kg
# v_vapor_sv = 60.0 * 461.5 * T_sv / P_sv         # exaggerated ideal-gas-like branch for clear plotting
v_vapor_sv = 461.5 * T_sv / P_sv         # exaggerated ideal-gas-like branch for clear plotting
# print(v_vapor_sv)

# --- solid-liquid boundary: schematic melting curve with negative slope for water ---
T_sl = np.linspace(T_triple - 22.0, T_triple + 1.0, 180)
dT = T_sl - T_triple

# Pressure increases as temperature decreases from triple point
# simple quadratic schematic, always >= triple pressure
P_sl = P_triple + 1.6e7 * (-dT / 22.0) + 0.6e7 * (dT / 22.0)**2
P_sl = np.maximum(P_sl, P_triple)

# schematic specific volumes for solid and liquid branches
v_solid_sl = 1.0 / 917.0 * np.ones_like(T_sl)   # ice
v_liquid_sl = 1.0 / 999.0 * np.ones_like(T_sl)  # liquid water
# v_vap


## 4. Quick 2D checks in \(P\)-\(T\) space


In [ ]:

plt.figure(figsize=(7, 5))
plt.semilogy(T_sv, P_sv, label="Solid-vapor")
plt.semilogy(T_sl, P_sl, label="Solid-liquid")
plt.semilogy(Tsat[:len(Psat)], Psat, label="Liquid-vapor")
plt.scatter([T_triple], [P_triple], label="Triple point")
plt.scatter([T_crit], [P_crit], label="Critical point")
plt.xlabel("Temperature [K]")
plt.ylabel("Pressure [Pa]")
plt.title("Water phase boundaries in P-T space")
plt.grid(True)
plt.legend()
plt.show()
# v_vap


## 5. Quick 2D checks in \(P\)-\(v\) space


In [ ]:

plt.figure(figsize=(8, 5))
plt.semilogx(v_solid_sv, P_sv / 1e6, label="Solid branch (solid-vapor)")
plt.semilogx(v_vapor_sv, P_sv / 1e6, label="Vapor branch (solid-vapor)")
plt.semilogx(v_solid_sl, P_sl / 1e6, label="Solid branch (solid-liquid)")
plt.semilogx(v_liquid_sl, P_sl / 1e6, label="Liquid branch (solid-liquid)")
plt.semilogx(v_liq, Psat / 1e6, label="Liquid branch (liquid-vapor)")
plt.semilogx(v_vap, Psat / 1e6, label="Vapor branch (liquid-vapor)")
plt.scatter([v_crit], [P_crit / 1e6], label="Critical point")
plt.xlabel("Specific volume v [m$^3$/kg]")
plt.ylabel("Pressure [MPa]")
plt.title("Water coexistence boundaries in P-v space")
plt.grid(True)
plt.legend(loc="best")
plt.show()
# v_vap


## 6. Interactive 3D PVT phase-boundary plot

We plot:
- $x = \log_{10}(v)$
- $y = T$
- $z = \log_{10}(P)$

This compresses the large dynamic range and makes the geometry easier to see.


In [ ]:

fig = go.Figure()

# solid-vapor branches
fig.add_trace(go.Scatter3d(
    x=np.log10(v_solid_sv),
    # x=v_solid_sv,
    y=T_sv,
    z=np.log10(P_sv),
    mode="lines",
    name="Solid branch (solid-vapor)"
))

fig.add_trace(go.Scatter3d(
    x=np.log10(v_vapor_sv),
    # x=v_vapor_sv,
    y=T_sv,
    z=np.log10(P_sv),
    mode="lines",
    name="Vapor branch (solid-vapor)"
))

# solid-liquid branches
fig.add_trace(go.Scatter3d(
    x=np.log10(v_solid_sl),
    # x=v_solid_sl,
    y=T_sl,
    z=np.log10(P_sl),
    mode="lines",
    name="Solid branch (solid-liquid)"
))

fig.add_trace(go.Scatter3d(
    x=np.log10(v_liquid_sl),
    # x=v_liquid_sl,
    y=T_sl,
    z=np.log10(P_sl),
    mode="lines",
    name="Liquid branch (solid-liquid)"
))

# liquid-vapor branches
fig.add_trace(go.Scatter3d(
    x=np.log10(v_liq),
    # x=v_liq,
    y=Tsat[:len(Psat)],
    z=np.log10(Psat),
    mode="lines",
    name="Liquid branch (liquid-vapor)"
))

fig.add_trace(go.Scatter3d(
    x=np.log10(v_vap),
    # x=v_vap,
    y=Tsat[:len(Psat)],
    z=np.log10(Psat),
    mode="lines",
    name="Vapor branch (liquid-vapor)"
))

# triple point
v_triple_s = 1.0 / 917.0
v_triple_l = 1.0 / PropsSI("D", "T", T_triple + 0.01, "Q", 0, fluid)
v_triple_v = 1.0 / PropsSI("D", "T", T_triple + 0.01, "Q", 1, fluid)

fig.add_trace(go.Scatter3d(
    x=np.log10([v_triple_s, v_triple_l, v_triple_v]),
    y=[T_triple, T_triple, T_triple],
    z=np.log10([P_triple, P_triple, P_triple]),
    mode="markers+text",
    text=["Triple point (solid)", "Triple point (liquid)", "Triple point (vapor)"],
    textposition="top center",
    name="Triple point"
))

# critical point
fig.add_trace(go.Scatter3d(
    x=[np.log10(v_crit)],
    y=[T_crit],
    z=[np.log10(P_crit)],
    mode="markers+text",
    text=["Critical point"],
    textposition="top center",
    name="Critical point"
))

fig.update_layout(
    title="Water PVT Phase Diagram with Solid–Liquid–Vapor Boundaries",
    scene=dict(
        xaxis_title="log10(v [m^3/kg])",
        yaxis_title="Temperature [K]",
        zaxis_title="log10(P [Pa])",
        camera=dict(eye=dict(x=1.8, y=1.5, z=1.1))
    ),
    width=980,
    height=800
)

fig.show()



## 7. Make a simple boundary surface patch near the triple point

To make the 3D figure look a little more like a textbook phase-boundary diagram,
we can add simple quadrilateral-like sheets between the branch curves.

These are still schematic.


In [ ]:

def add_ruled_surface(fig, x1, y1, z1, x2, y2, z2, name):
    x = np.vstack([x1, x2])
    y = np.vstack([y1, y2])
    z = np.vstack([z1, z2])

    fig.add_trace(go.Surface(
        x=x,
        y=y,
        z=z,
        showscale=False,
        opacity=0.45,
        name=name,
        hoverinfo="skip"
    ))

fig2 = go.Figure(fig)

# Add surface sheets
add_ruled_surface(
    fig2,
    np.log10(v_solid_sv), T_sv, np.log10(P_sv),
    np.log10(v_vapor_sv), T_sv, np.log10(P_sv),
    "Solid-vapor sheet"
)

add_ruled_surface(
    fig2,
    np.log10(v_solid_sl), T_sl, np.log10(P_sl),
    np.log10(v_liquid_sl), T_sl, np.log10(P_sl),
    "Solid-liquid sheet"
)

add_ruled_surface(
    fig2,
    np.log10(v_liq), Tsat[:len(Psat)], np.log10(Psat),
    np.log10(v_vap), Tsat[:len(Psat)], np.log10(Psat),
    "Liquid-vapor sheet"
)

fig2.update_layout(
    title="Water PVT Diagram with Schematic Phase-Boundary Sheets",
    scene=dict(
        xaxis_title="log10(v [m^3/kg])",
        yaxis_title="Temperature [K]",
        zaxis_title="log10(P [Pa])",
        camera=dict(eye=dict(x=1.8, y=1.5, z=1.1))
    ),
    width=980,
    height=800
)

fig2.show()



## 8. Export to HTML

```python
fig2.write_html("water_pvt_with_solid_boundaries.html")
```

Then open the file in a browser and rotate it.



## 9. Remarks

This notebook fixes the main limitation of the earlier version: it now shows

- **solid–vapor**
- **solid–liquid**
- **liquid–vapor**

boundaries explicitly.

Again, the **liquid–vapor** boundary is based on CoolProp, while the **solid-related** boundaries are schematic so the overall 3D structure is visible and easy to teach.

If you want, the next step would be to make a more textbook-style figure with:
- colored phase regions labeled **solid**, **liquid**, **vapor**
- translucent phase surfaces
- a more cube-like perspective similar to standard PVT diagrams
- annotations for **isothermal** and **constant-pressure** lines
